# 5. 深度学习计算（上）

## 5.1 层和块

> 之前的代码中，我们仅使用过 `nn.Sequential` 这个线性容器 (无法处理条件分支逻辑)
>
> 为了实现更复杂的架构，我们需要学习 PyTorch 的核心基类：<b>nn.Module<b>。

### 1. 核心概念：块 (Blocks)

在 PyTorch 中，**“块”**（Block）可以描述单个层、由多个层组成的组件，甚至整个模型本身。
*   **数学本质**：块是一个将输入张量转化为输出张量的函数。
*   **工程本质**：块是一个继承自 `nn.Module` 的类，它维护着**状态**（参数）和**行为**（前向传播）。

---


### 2. 自定义多层感知机块

In [2]:
import torch
from torch import nn, Tensor

class MLP(nn.Module):
    """自定义多层感知机块。
    
    展示了 nn.Module 的基本结构：__init__ 负责定义组件，forward 负责定义计算流。
    """
    def __init__(self, input_dim: int, hidden_dim: int, output_dim: int):
        """初始化模型层。
        
        Args:
            input_dim: 输入特征维度。
            hidden_dim: 隐藏层神经元数量。
            output_dim: 输出类别数量。
        """
        # 必须调用父类的 __init__(), 否则无法自动注册参数
        super().__init__()

        # 将层定义为类的成员属性
        self.hidden = nn.Linear(input_dim, hidden_dim)
        self.out = nn.Linear(hidden_dim, output_dim)
        self.relu = nn.ReLU()

    def forward(self, X: Tensor) -> Tensor:
        """定义前向传播逻辑。
        
        Args:
            X: 输入向量。

        Returns:
            Tensor: 输出向量。
        """
        # 我们可以自由控制计算顺序，甚至可以在这里加入 Python 的 if/for 逻辑
        h = self.relu(self.hidden(X))
        return self.out(h)
    
# --- 使用示例 ---
net = MLP(input_dim=20, hidden_dim=256, output_dim=10)
X = torch.randn(2, 20)
# nn.Module 的魔法方法 __call__ 允许调用 net(X) 时执行 forward 逻辑
# 建议：始终使用 net(X) 而不是 net.forward(X), 因为直接调用 forward 可能会导致某些监控或调试工具失效。
output = net(X) 
print(f"自定义的 MLP 输出形状: {output.shape}")

自定义的 MLP 输出形状: torch.Size([2, 10])


---

### 3. 细致梳理：`nn.Module` 

为什么继承 `nn.Module` 后，模型就能自动求导并管理参数了？

1.  **参数注册 (Parameter Registration)**：
    当你执行 `self.hidden = nn.Linear(...)` 时，`nn.Module` 会在后台自动识别出这是一个包含参数的层，并将其权重和偏置加入到模型的 `parameters()` 列表中。
2.  **`__call__` 与 `forward`**：
    在 PyTorch 中，你调用 `net(X)` 而不是 `net.forward(X)`。这是因为父类实现了 `__call__` 方法，它在运行 `forward` 之前和之后会执行一些必要的“钩子”操作（如注册梯度）。
3.  **递归性**：
    一个 `nn.Module` 可以包含另一个 `nn.Module`。这允许我们像搭乐高一样，把小组件拼成大系统。

---


### 4. 实现一个自定义的顺序块 (Sequential)

In [3]:
class MySequential(nn.Module):
    """手动实现 Sequential 容器。"""
    def __init__(self, *args: nn.Module):
        super().__init__()
        # _modules 是父类 nn.Module 内部维护的一个有序字典
        for idx, module in enumerate(args):
            self._modules[str(idx)] = module

    def forward(self, X: Tensor) -> Tensor:
        # 按顺序执行每一个已注册的块
        for block in self._modules.values():
            X = block(X)
        return X
    
# --- 使用方法 ---
net_test = MySequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))
print(f"MySequential 输出形状: {net_test(torch.randn(2, 20)).shape}")

MySequential 输出形状: torch.Size([2, 10])


---

### 5. 在前向传播中执行复杂计算 (Fixed Parameters)

> 这是 nn.Sequential 无法做到的事：在计算过程中加入不参与训练的常数或复杂的控制流。


In [4]:
class FixedMLP(nn.Module):
    """展示在 forward 中加入复杂计算逻辑。"""
    def __init__(self):
        super().__init__()
        # rand_weight 是随机生成的，且不设为 Parameter, 因此不会被梯度更新
        self.rand_weight = torch.randn(20, 20)
        self.linear = nn.Linear(20, 20)

    def forward(self, X: Tensor) -> Tensor:
        X = self.linear(X)
        # 使用常数权重进行计算，并加入复杂的 Python 控制流
        X = torch.relu(X @ self.rand_weight + 1)

        # 复用同一个线性层 (参数共享)
        X = self.linear(X)

        # 控制流演示
        while X.abs().sum() > 1:
            X /= 2
        return X.sum()

# --- 使用方法 ---
net_current = FixedMLP()
X = torch.randn(2, 20)
output = net_current(X)

print(f"1. 输出结果: {output.item():.4f} (是一个标量)")

# 演示 1：证明 rand_weight 不在参数列表中（不会被训练）
params = [name for name, _ in net_current.named_parameters()]
print(f"2. 模型参数列表: {params}")
print(f"   注意：'rand_weight' 不在其中，因为它只是一个普通的 Tensor 而非 nn.Parameter")

# 演示 2：执行反向传播，查看梯度
output.backward()
print(f"3. linear.weight 的梯度形状: {net_current.linear.weight.grad.shape}")

# 演示 3：由于 forward 中有 while 循环，我们可以检查输出是否符合预期（sum < 1）
print(f"4. 最终 X.sum() 是否小于 1: {output < 1}")

1. 输出结果: -0.5927 (是一个标量)
2. 模型参数列表: ['linear.weight', 'linear.bias']
   注意：'rand_weight' 不在其中，因为它只是一个普通的 Tensor 而非 nn.Parameter
3. linear.weight 的梯度形状: torch.Size([20, 20])
4. 最终 X.sum() 是否小于 1: True


In [ ]:
# 嵌套块练习
class NestMLP(nn.Module):
    """嵌套块练习：级联(串联)两个 MLP。"""
    def __init__(self):
        super().__init__()
        # 在容器中嵌套我们自定义的 MLP 块
        self.net = nn.Sequential(
            MLP(20, 64, 32),
            MLP(32, 16, 10)
        )

    def forward(self, X: Tensor) -> Tensor:
        return self.net(X)
    
# --- 测试 ---
nest_net = NestMLP()
print(f"NestMLP 输出形状: {nest_net(torch.randn(2, 20)).shape}")

NestMLP 输出形状: torch.Size([2, 10])


### 6. 关于“块”的工程规范建议

**`nn.Module`** 的开发准则：

1.  **参数注册**：永远在 `__init__` 中将子模块赋值给 `self.xxx`，这样 `net.parameters()` 才能找到它们。
2.  **逻辑位置**：所有的参数初始化逻辑放在 `__init__`，所有的计算逻辑放在 `forward`。**严禁**在 `forward` 中定义新的 `nn.Linear`（否则每次运行都会创建新权重）。
3.  **类型注解**：在 `forward` 中明确标注 `Tensor` 的输入输出，这对于大型模型 Debug 至关重要。

---
